# RAG Pipeline — Development Notebook

Exploratory notebook used to prototype the ingestion pipeline, retrieval chain, and RAGAS evaluation for the ASSUME documentation corpus.

## Setup

```bash
pip install langchain langchain-community langchain-huggingface langchain-qdrant langchain-openai \
    Markdown unstructured langchain-text-splitters pypandoc ragas ipywidgets pdfminer.six \
    pi_heif unstructured_inference pdf2image pytesseract unstructured-pytesseract \
    sentence-transformers pymongo rank-bm25 fastembed
```

```bash
sudo apt-get install tesseract-ocr
```

In [ ]:
from langchain_community.document_loaders import DirectoryLoader, UnstructuredMarkdownLoader, UnstructuredPDFLoader
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_qdrant import QdrantVectorStore
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
from langchain_community.cross_encoders import HuggingFaceCrossEncoder


In [ ]:
from dotenv import load_dotenv

load_dotenv() 
print("Loaded .env file")

In [ ]:
import os

QDRANT_URL  = "http://localhost:6333"
COLLECTION  = "assume_docs_elements_small_chunks_pc_v2_sparse_code"

EMBED_BASE_URL = "https://chat.kiconnect.nrw/api/v1"
EMBED_MODEL    = "qwen-qwen3-embedding-8b"

LLM_BASE_URL = "https://chat.kiconnect.nrw/api/v1"
LLM_MODEL    = "openai-gpt-oss-120b"

DATA_DIR = "data/"

API_KEY = os.getenv("API_KEY")

QUESTION = "How does the reinforcement learning work in ASSUME?"


In [ ]:
LLM_MODEL_INPUT_TOKENS_USED = 0
LLM_MODEL_OUTPUT_TOKENS_USED = 0
LLM_MILLISECONDS_TAKEN = 0.0

DB_QUERIE_MILLISECONDS_TAKEN = 0.0
LOCAL_PROCESSING_MILLISECONDS_TAKEN = 0.0


## Data Loading

Load markdown docs, PDFs, and Python source files from the ASSUME repository.

In [ ]:
loader = DirectoryLoader(DATA_DIR, glob="full-md-docs/**/*.md", loader_cls=UnstructuredMarkdownLoader, loader_kwargs={ "mode": "elements"}, show_progress=True)
docs   = loader.load()
print(f"Loaded {len(docs)} documents")


In [ ]:
pdf_loader = DirectoryLoader(
    DATA_DIR,
    glob="**/*.pdf",
    loader_cls=UnstructuredPDFLoader,
    loader_kwargs={
        "mode": "elements",
        "strategy": "hi_res",
        "languages": ["eng"],
        "coordinates": True,  # needed for multi-column layouts
        "infer_table_structure": True,
    },
    show_progress=True,
)

"""
hi_res sucht nach:
- Überschriften
- Absätze
- Tabellen
- Listen
- Spalten
- Bildbereiche
- Seitenelemente
"""

pdf_docs = pdf_loader.load()

print(f"Loaded {len(pdf_docs)} PDF documents")

In [ ]:
CODE_DIR = "/home/x46/Documents/Uni/Master/REG KI/Sek/assume-main" 

code_loader = DirectoryLoader(
    CODE_DIR,
    glob="**/*.py",
    loader_cls=TextLoader,
    loader_kwargs={
        "encoding": "utf-8",
    },
    recursive=True,
    silent_errors=True,
    show_progress=True,
)

code_docs = code_loader.load()

# Quelle als Metadaten kennzeichnen
for doc in code_docs:
    doc.metadata["doc_type"] = "code"

print(f"Loaded {len(code_docs)} Python source files")

In [ ]:
full_docs = docs + pdf_docs + code_docs
print(f"Total documents (MD + PDF + Python): {len(full_docs)}")


## Chunking & Parent-Child Indexing

Split documents into small chunks for dense retrieval. Each chunk carries a `parent_id` pointing back to the full section it came from — used later to fetch the full context when answering.

In [ ]:
import hashlib

def split_documents_by_title(docs):
    sections = []
    current_section = None

    for doc in docs:
        category = doc.metadata.get("category")
        text = doc.page_content.strip()

        if not text:
            continue

        if category == "Title":
            current_section = {
                "title": text,
                "documents": [doc],
                "text": text,
                "metadata": {
                    "source": doc.metadata.get("source"),
                    "page_start": doc.metadata.get("page_number"),
                    "title_element_id": doc.metadata.get("element_id"),
                },
            }
            sections.append(current_section)

        else:
            if current_section is None:
                current_section = {
                    "title": "Ohne Titel",
                    "documents": [],
                    "text": "",
                    "metadata": {
                        "source": doc.metadata.get("source"),
                        "page_start": doc.metadata.get("page_number"),
                        "title_element_id": None,
                    },
                }
                sections.append(current_section)

            current_section["documents"].append(doc)
            current_section["text"] += "\n\n" + text

    return sections


def get_parent_text_from_elements(elements):
    # Python-Code-Dateien separat behandeln: kein category/element_id von Unstructured
    code_elements = [e for e in elements if e.metadata.get("doc_type") == "code"]
    structured_elements = [e for e in elements if e.metadata.get("doc_type") != "code"]

    res = []

    # Python Code: jede Datei = eine Parent-Sektion mit stabilem element_id (MD5 des Pfads)
    for doc in code_elements:
        source = doc.metadata.get("source", "unknown")
        eid = hashlib.md5(source.encode()).hexdigest()
        res.append({
            "page_content": doc.page_content,
            "source": source,
            "element_id": eid,
            "child_element_ids": [],
        })
    if code_elements:
        print(f"Processed {len(code_elements)} Python code files as parent sections.")

    # Strukturierte Dokumente (MD/PDF via Unstructured) nach Titel gruppieren
    source_docs = {}
    for element in structured_elements:
        source = element.metadata.get("source")
        if source not in source_docs:
            source_docs[source] = []
        source_docs[source].append(element)

    for source, elems in source_docs.items():
        split = split_documents_by_title(elems)
        for section in split:
            res.append(
                {
                    "page_content": section["text"],
                    "source": section["metadata"]["source"],
                    "element_id": section["metadata"]["title_element_id"],
                    "child_element_ids": [e.metadata.get("element_id") for e in section["documents"]],
                }
            )
        print(f"Processed {len(elems)} elements from source {source} into {len(split)} sections.")
    return res


parent_elements = get_parent_text_from_elements(full_docs)

In [ ]:
from transformers import AutoTokenizer
hf_tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-Embedding-8B")

In [ ]:
splitter = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(
    hf_tokenizer,
    chunk_size=320,
    chunk_overlap=0,
    separators=["\n\n", "\n", ". ", " ", ""],
)
chunks = splitter.split_documents(full_docs)

# Build reverse map: child element_id -> parent element_id
child_to_parent: dict[str, str] = {}
for pe in parent_elements:
    pid = pe["element_id"]
    if not pid:
        continue
    child_to_parent[pid] = pid  # title element maps to itself
    for ceid in pe.get("child_element_ids", []):
        if ceid:
            child_to_parent[ceid] = pid

# Annotate every chunk with parent_id so retrieval is uniform for all doc types
for chunk in chunks:
    if chunk.metadata.get("doc_type") == "code":
        source = chunk.metadata.get("source", "")
        chunk.metadata["parent_id"] = hashlib.md5(source.encode()).hexdigest()
    else:
        eid = chunk.metadata.get("element_id")
        chunk.metadata["parent_id"] = child_to_parent.get(eid) if eid else None

annotated = sum(1 for c in chunks if c.metadata.get("parent_id"))
print(f"Total chunks: {len(chunks)}  |  with parent_id: {annotated}")


## Ingestion

> **First-run only.** Indexes chunks into Qdrant (dense + sparse BM25 vectors) and parent sections into MongoDB.

In [ ]:
from langchain_qdrant import FastEmbedSparse, RetrievalMode

# Qdrant/bm25 is a FastEmbed BM25 sparse model
# It generates sparse vectors that are stored alongside dense vectors in the same collection.
sparse_embeddings = FastEmbedSparse(model_name="Qdrant/bm25")
print("Sparse embeddings (Qdrant/bm25 via FastEmbed) ready")


In [ ]:
embeddings = OpenAIEmbeddings(
    model=EMBED_MODEL,
    base_url=EMBED_BASE_URL,
    api_key=API_KEY,
    check_embedding_ctx_length=False,
)
vectorstore = QdrantVectorStore.from_documents(
    documents=chunks,
    embedding=embeddings,
    sparse_embedding=sparse_embeddings,
    url=QDRANT_URL,
    collection_name=COLLECTION,
    retrieval_mode=RetrievalMode.HYBRID,
)
print("Indexed into Qdrant (dense + sparse BM25 vectors)")


In [ ]:
from pymongo import MongoClient, UpdateOne

MONGO_URI = os.getenv("MONGO_URI", "mongodb://localhost:27017")
MONGO_DB = os.getenv("MONGO_DB", "rag")

client = MongoClient(MONGO_URI)
db = client[MONGO_DB]

mongo_collection = db[COLLECTION] 


In [ ]:
# Create index on element_id (non-unique)
mongo_collection.create_index("element_id", name="idx_element_id")

ops = []
for item in parent_elements:
    doc = dict(item)
    eid = doc.get("element_id")

    if eid:
        ops.append(
            UpdateOne(
                {"element_id": eid},
                {"$set": doc},
                upsert=True,
            )
        )
    else:
        print(f"Warning: Skipping document without element_id: {doc.get('metadata', {}).get('source', 'unknown source')}")

if ops:
    result = mongo_collection.bulk_write(ops, ordered=False)
    print(
        f"Upserted: {result.upserted_count}, "
        f"Modified: {result.modified_count}, "
        f"Matched: {result.matched_count}"
    )

print(f"MongoDB collection ready: {MONGO_DB}.{COLLECTION}")
print("Index info:", mongo_collection.index_information())

## Retrieval Pipeline

Hybrid retrieval (dense + BM25) → multi-query rewriting → cross-encoder reranking → parent-document fetch from MongoDB.

In [ ]:
embeddings = OpenAIEmbeddings(
    model=EMBED_MODEL,
    base_url=EMBED_BASE_URL,
    api_key=API_KEY,
    check_embedding_ctx_length=False,
)
# Main hybrid store: Qdrant fuses dense + sparse internally (RRF)
vectorstore = QdrantVectorStore.from_existing_collection(
    embedding=embeddings,
    sparse_embedding=sparse_embeddings,
    url=QDRANT_URL,
    collection_name=COLLECTION,
    retrieval_mode=RetrievalMode.HYBRID,
)



In [ ]:
from langchain_core.documents import Document

def reciprocal_rank_fusion(
    doc_lists: list[list[Document]],
    weights: list[float] | None = None,
    k: int = 60,
) -> list[tuple[Document, float]]:
    """
    RRF: promotes documents that appear at high ranks across multiple result lists.
    Each document's score = Σ  w_i / (rank_i + k).
    weights: per-list multipliers (default 1.0 for every list).
    k:       smoothing constant (classic value: 60).
    Returns list of (Document, rrf_score) sorted descending by score.
    """
    if weights is None:
        weights = [1.0] * len(doc_lists)

    scores: dict[tuple, float] = {}
    doc_map: dict[tuple, Document] = {}

    for w, doc_list in zip(weights, doc_lists):
        for rank, doc in enumerate(doc_list):
            key = (doc.metadata.get("source", ""), doc.page_content[:300])
            if key not in doc_map:
                doc_map[key] = doc
                scores[key] = 0.0
            scores[key] += w / (rank + k)

    return [(doc_map[key], scores[key]) for key in sorted(scores, key=scores.__getitem__, reverse=True)]



In [ ]:
hybrid_retriever = vectorstore.as_retriever(
        search_kwargs={"k": 25}
)
print("Hybrid retriever: Qdrant native hybrid search (dense + sparse)  →  RRF  (k=25)")

In [ ]:
# Cross-Encoder rerankt die Top-k Kandidaten neu und gibt nur die besten top_n weiter
_cross_encoder = HuggingFaceCrossEncoder(model_name="cross-encoder/ms-marco-MiniLM-L-6-v2")
_compressor = CrossEncoderReranker(model=_cross_encoder, top_n=10)


In [ ]:
prompt = ChatPromptTemplate.from_template("""You are an expert assistant for the ASSUME energy simulation framework.
Answer the question using ONLY the information provided in the context below.
If the context does not contain enough information to answer, say "I don't have enough information to answer this question."
Do not make up or infer facts beyond what is explicitly stated in the context.

Context:
{context}

Question: {question}

Answer concisely and precisely:""")

In [ ]:
import time
from langchain_core.callbacks import BaseCallbackHandler
from langchain_core.outputs import LLMResult


class MetricsCallbackHandler(BaseCallbackHandler):
    def on_llm_start(self, serialized, prompts, **kwargs):
        self._llm_start_time = time.time()

    def on_llm_end(self, response: LLMResult, **kwargs):
        global LLM_MILLISECONDS_TAKEN, LLM_MODEL_INPUT_TOKENS_USED, LLM_MODEL_OUTPUT_TOKENS_USED
        LLM_MILLISECONDS_TAKEN += (time.time() - self._llm_start_time) * 1000
        usage = (response.llm_output or {}).get("token_usage", {})
        LLM_MODEL_INPUT_TOKENS_USED += usage.get("prompt_tokens", 0)
        LLM_MODEL_OUTPUT_TOKENS_USED += usage.get("completion_tokens", 0)


_metrics_handler = MetricsCallbackHandler()
print("Metrics callback handler ready")


In [ ]:
llm = ChatOpenAI(
    model=LLM_MODEL,
    base_url=LLM_BASE_URL,
    api_key=API_KEY,
    callbacks=[_metrics_handler],
)


In [ ]:
import json
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

max_queries = 5


query_rewrite_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """You are an expert query planner and query rewriter for a RAG system.

The user may ask questions whose answer is not explicitly stated in one passage. Your task is to generate diverse search queries that retrieve the evidence needed to answer the user's question accurately.

Use the user's original question and the optional context below:

Additional context:
{pre_context}

Generate several clearly different search queries that cover the following retrieval strategies:

1. Direct query  
   Closely rephrase the original question while preserving its exact intent.

2. Keyword query  
   Create a short keyword-based query using the most important terms.

3. Concept queries  
   Search for key concepts, related ideas, synonyms, or implied themes from the question.

4. Mechanism queries  
   Search for causes, reasons, effects, consequences, mechanisms, or relationships relevant to the question.

5. Technical or domain-specific queries  
   Use terminology from the relevant domain, especially when applicable: electricity market simulation, agent-based modeling, reinforcement learning, market design, policy support, validation, interpretability, transparency, or explainability.

6. Alternative-perspective queries  
   Reformulate the question from a different but still relevant perspective, such as evaluation, policy relevance, model behavior, stakeholder use, or decision support.

7. Negative or risk queries  
   If useful, generate queries that retrieve risks, limitations, or failure modes related to the question, such as market power, price manipulation, black-box behavior, invalid assumptions, lack of validation, biased outcomes, or poor interpretability.

8. Abbreviation/entity queries  
   If useful, include abbreviations, key entities, technical terms, or domain-specific shorthand that may appear in documents.

Rules:
- Preserve the original intent exactly.
- Do not invent an answer.
- Do not invent facts, names, numbers, entities, or assumptions.
- Do not create irrelevant or overly broad queries.
- Use the additional context only to improve retrieval, not to change the question.
- Queries may use related concepts, synonyms, and domain terminology.
- Make sure the queries are clearly different from each other.
- Return only a valid JSON list of strings.
- Do not include explanations, Markdown, comments, or text outside the JSON list.
"""
    ),
    (
        "human",
        """Question:
{question}

Number of variants:
{num_queries}"""
    )
])

query_rewriter = (
    query_rewrite_prompt
    | llm
    | StrOutputParser()
)


def parse_queries(raw: str, original_question: str) -> list[str]:
    """
    Parsed die LLM-Ausgabe robust und fügt die Originalfrage immer hinzu.
    """
    try:
        queries = json.loads(raw)
        if not isinstance(queries, list):
            queries = []
    except Exception:
        queries = []

    queries = [q.strip() for q in queries if isinstance(q, str) and q.strip()]

    # Originalfrage immer mit aufnehmen
    queries = [original_question] + queries

    # Deduplizieren, Reihenfolge erhalten
    seen = set()
    unique_queries = []
    for q in queries:
        key = q.lower()
        if key not in seen:
            seen.add(key)
            unique_queries.append(q)

    return unique_queries[:max_queries]


def retrieve_multi_query(question: str):
    """
    Query-Rewriting + Multi-Query Retrieval mit RRF-Fusion.

    Für jede Query-Variante werden Dokumente über den Hybrid-Retriever
    (dense + sparse) + Cross-Encoder abgerufen.  Die pro-Query-Ergebnislisten
    werden anschließend via Reciprocal Rank Fusion (RRF) zusammengeführt:
    Dokumente, die in mehreren Query-Varianten hoch gerankt werden,
    steigen im kombinierten Ranking auf.
    """
    global DB_QUERIE_MILLISECONDS_TAKEN, LOCAL_PROCESSING_MILLISECONDS_TAKEN

    # Initial retrieval for query-rewriting context (DB + local reranking timed separately)
    t0 = time.time()
    basic_raw = hybrid_retriever.invoke(question)
    DB_QUERIE_MILLISECONDS_TAKEN += (time.time() - t0) * 1000

    t0 = time.time()
    basic_info = _compressor.compress_documents(basic_raw, question)
    LOCAL_PROCESSING_MILLISECONDS_TAKEN += (time.time() - t0) * 1000

    print(f"Basic info for query rewriting:", basic_info)

    raw_queries = query_rewriter.invoke({"question": question, "num_queries": max_queries, "pre_context": basic_info})
    queries = parse_queries(raw_queries, question)

    # Collect one result list per query variant
    all_doc_lists: list[list] = []
    print(queries)

    for q in queries:
        t0 = time.time()
        docs_raw = hybrid_retriever.invoke(q)  # holt 25 Dokumente pro Query-Variante
        DB_QUERIE_MILLISECONDS_TAKEN += (time.time() - t0) * 1000

        t0 = time.time()
        docs = _compressor.compress_documents(docs_raw, q)  # reranked, best first
        LOCAL_PROCESSING_MILLISECONDS_TAKEN += (time.time() - t0) * 1000

        all_doc_lists.append(docs)

    # RRF across query variants: documents appearing in multiple lists
    # and at high ranks get a boosted combined score
    # Returns (doc, rrf_score) tuples - scores are preserved for downstream parent selection
    return reciprocal_rank_fusion(all_doc_lists, k=40)


multi_query_compression_retriever = RunnableLambda(retrieve_multi_query)


In [ ]:
from langchain_core.documents import Document


def fetch_top_parent_from_mongo(scored_docs):
    """
    scored_docs: list of (Document, rrf_score) from reciprocal_rank_fusion.
    Every chunk carries parent_id in its metadata (set at index time).
    Aggregates RRF scores per parent and returns the top-10 parent documents.
    """
    global DB_QUERIE_MILLISECONDS_TAKEN

    parent_id_scores: dict[str, float] = {}

    for doc, rrf_score in scored_docs:
        parent_id = doc.metadata.get("parent_id")
        if parent_id:
            parent_id_scores[parent_id] = parent_id_scores.get(parent_id, 0.0) + rrf_score

    top_parent_ids = sorted(parent_id_scores, key=parent_id_scores.__getitem__, reverse=True)[:10]
    print(f"Top parent IDs: {top_parent_ids}")

    parent_docs = []
    for pid in top_parent_ids:
        t0 = time.time()
        parent = mongo_collection.find_one(
            {"element_id": pid},
            {"page_content": 1, "element_id": 1, "source": 1, "_id": 0},
        )
        DB_QUERIE_MILLISECONDS_TAKEN += (time.time() - t0) * 1000

        if parent:
            parent_docs.append(
                Document(
                    page_content=parent["page_content"],
                    metadata={
                        "element_id": parent["element_id"],
                        "retrieval": "parent",
                        "source": parent.get("source", "unknown"),
                    },
                )
            )

    return parent_docs


def retrieve_multi_query_with_parents(question: str):
    scored_chunks = retrieve_multi_query(question)  # list of (Document, rrf_score)
    return fetch_top_parent_from_mongo(scored_chunks)


multi_query_parent_retriever = RunnableLambda(retrieve_multi_query_with_parents)
print("Parent retriever ready")


In [ ]:
from langchain_core.runnables import RunnableParallel

# Chain that produces only the answer string
_answer_chain = (
        prompt
    | llm
    | StrOutputParser()
)

# Chain that returns answer + source documents
chain_with_sources = (
    RunnableParallel({"context": multi_query_parent_retriever, "question": RunnablePassthrough()})
    .assign(answer=_answer_chain)
)

# Convenience: plain chain answer only for evaluation
chain = chain_with_sources | (lambda x: x["answer"])


In [ ]:
LLM_MODEL_INPUT_TOKENS_USED = 0
LLM_MODEL_OUTPUT_TOKENS_USED = 0
LLM_MILLISECONDS_TAKEN = 0.0

DB_QUERIE_MILLISECONDS_TAKEN = 0.0
LOCAL_PROCESSING_MILLISECONDS_TAKEN = 0.0

result = chain_with_sources.invoke("How does the scenario loader work in ASSUME?")

print("\n── Answer ──────────────────────────────────────────")
print(result["answer"])

print("\n── Sources ─────────────────────────────────────────", len(result["context"]))
unique_sources = set()
for doc in result["context"]:
    source = doc.metadata.get("source", "unknown")
    if source not in unique_sources:
        unique_sources.add(source)
        print(f"[{len(unique_sources)}] {source}")

print(f"\nLLM_MODEL_INPUT_TOKENS_USED: {LLM_MODEL_INPUT_TOKENS_USED}")
print(f"LLM_MODEL_OUTPUT_TOKENS_USED: {LLM_MODEL_OUTPUT_TOKENS_USED}")
print(f"LLM_MILLISECONDS_TAKEN: {LLM_MILLISECONDS_TAKEN}")
print(f"DB_QUERIE_MILLISECONDS_TAKEN: {DB_QUERIE_MILLISECONDS_TAKEN}")
print(f"LOCAL_PROCESSING_MILLISECONDS_TAKEN: {LOCAL_PROCESSING_MILLISECONDS_TAKEN}")

## Evaluation

RAGAS evaluation on a fixed set of questions using Faithfulness and Answer Relevancy metrics.

In [ ]:
from ragas import EvaluationDataset, SingleTurnSample, evaluate
from ragas.metrics._faithfulness import Faithfulness
from ragas.metrics._answer_relevance import AnswerRelevancy
from ragas.run_config import RunConfig
from ragas.llms import llm_factory
from openai import OpenAI
import pandas as pd

_openai_client = OpenAI(base_url=LLM_BASE_URL, api_key=API_KEY)

ragas_llm = llm_factory(
    'gpt-5.2',#LLM_MODEL,
    client=_openai_client,
)

class EmbeddingsAdapter:
    def __init__(self, lc_embeddings):
        self._emb = lc_embeddings

    def embed_query(self, text):
        return self._emb.embed_query(text)

    def embed_documents(self, texts):
        return self._emb.embed_documents(texts)

ragas_embeddings = EmbeddingsAdapter(embeddings)

# Conservative runtime settings for unstable endpoints.
run_config = RunConfig(max_workers=1, max_retries=6, timeout=240)


In [ ]:
# 1) Build evaluation samples from your existing RAG chain
eval_questions = [
    "How does the reinforcement learning work in ASSUME?",
    "What market clearing algorithms does ASSUME support?",
    "Which bidding strategies does ASSUME support?",
    "What are the main components of the ASSUME framework?",
    "How does the scenario loader work in ASSUME?",
    "Why is model transparency important when simulation results are used to support electricity market design decisions?",
    "How can ASSUME be extended to represent complex order types such as regular block orders and linked orders, and why does this matter for realistic day-ahead market simulations?",
    "What are the main differences between optimization-based, equilibrium-based, and agent-based approaches for electricity market modeling?",
    "Why might rule-based bidding strategies be insufficient for simulating future electricity markets with high renewable penetration?",
]


compression_retriever = ContextualCompressionRetriever(
    base_compressor=_compressor,
    base_retriever=hybrid_retriever,
)

samples = []
for q in eval_questions:
    retrieved_docs = compression_retriever.invoke(q)
    ctx_texts = [d.page_content for d in retrieved_docs]
    ans = chain.invoke(q)

    samples.append(
        SingleTurnSample(
            user_input=q,
            response=str(ans),
            retrieved_contexts=ctx_texts,
        )
    )
    print(f"✓ {q[:60]}...")

print(f"\nCollected {len(samples)} samples.")


In [ ]:
dataset = EvaluationDataset(samples=samples)

metrics = [
    Faithfulness(llm=ragas_llm),
    AnswerRelevancy(
        llm=ragas_llm,
        embeddings=ragas_embeddings,
        strictness=1,
    ),
]

result = evaluate(
    dataset=dataset,
    metrics=metrics,
    run_config=run_config,
    batch_size=1,
    raise_exceptions=False,
)

results_df = result.to_pandas()

overall_scores = {
    "faithfulness": results_df["faithfulness"].mean(skipna=True),
    "answer_relevancy": results_df["answer_relevancy"].mean(skipna=True),
}

In [ ]:
print("Overall:", overall_scores)
results_df